
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif;">

<div style="background: #1B5162; color: white; border-radius: 8px; padding: 28px 32px; text-align: center; position: relative;">
  <div style="font-size: 14pt; font-weight: 600; text-transform: uppercase; letter-spacing: 1px; opacity: 0.85; margin-bottom: 8px;">Lesson 08</div>
  <div style="font-size: 24pt; font-weight: 700; line-height: 1.3;">Ingest Data with CTAS and the Upload UI</div>
  <div style="font-size: 14pt; margin-top: 12px; opacity: 0.9;">Create tables from files using two methods: CREATE TABLE AS SELECT with format options and the Catalog Explorer upload interface.</div>
</div>

</div>

## REQUIRED — SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 14px 18px; border-radius: 4px; margin: 16px 0;">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before running this notebook, confirm your compute environment at the top-right of the notebook.

- Click the compute dropdown and select **Serverless** (the default option).
- If you do not see Serverless available, contact your workspace administrator.

**Note:** This notebook was developed and tested on **Serverless compute**. Other compute options may work but are not guaranteed to behave the same.
  </div>
</div>

### Setup
Run the cell below to configure your environment.

In [0]:
%run ./Includes/Classroom-Setup-1

**In this lesson:** You'll learn different ways to get data *into* tables.


<!-- LEARN: Two ingestion methods -->
<!-- Template: 2-card-colored-header-guidance -->

<div style="max-width: 950px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<div style="font-size: 20pt; font-weight: 700; color: #0b2026; margin-bottom: 6px;">Two Ways to Create a Table from a File</div>
<div style="font-size: 14pt; color: #5A6F77; margin-bottom: 24px;">Both methods produce a Delta table, but they serve different purposes.</div>

<div style="display: flex; gap: 20px; justify-content: center;">

<!-- Card 1: CTAS -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden; background: white;">
  <div style="background: #4299E0; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 18pt; font-weight: bold;">CREATE TABLE AS SELECT</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
      Write SQL to read a file and define exactly which columns to include, what format options to apply, and how the table should be structured. Repeatable and version-controlled.
    </div>
    <div style="background: rgba(66,153,224,0.10); border-left: 4px solid #4299E0; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
      <strong>Use when:</strong> You're building a pipeline, need reproducibility, or want to select specific columns from the source file.
    </div>
  </div>
</div>

<!-- Card 2: Upload UI -->
<div style="flex: 1; border: 2px solid #e0e0e0; border-radius: 12px; overflow: hidden; background: white;">
  <div style="background: #00A972; color: white; padding: 14px 20px; text-align: center;">
    <div style="font-size: 18pt; font-weight: bold;">Catalog Explorer Upload</div>
  </div>
  <div style="padding: 18px 20px;">
    <div style="font-size: 14pt; color: #555; line-height: 1.6; margin-bottom: 14px;">
      Drag and drop a file in the Catalog Explorer UI. Databricks auto-detects the format, infers the schema, and creates the table. No code required.
    </div>
    <div style="background: rgba(0,169,114,0.10); border-left: 4px solid #00A972; padding: 10px 12px; border-radius: 6px; font-size: 14pt;">
      <strong>Use when:</strong> Someone hands you a CSV and you need it in a table fast. Quick, ad-hoc imports where reproducibility is not a concern.
    </div>
  </div>
</div>

</div>

</div>

##### EXPAND FOR ADDITIONAL NOTES

<details>

**Choosing between the two methods**

- **CTAS** is the workhorse of data engineering. You write SQL that reads from a source (file, another table, or an external system), optionally transforms the data, and writes the result as a new table. Because it's code, it's repeatable: you can re-run it, put it in a pipeline, and version-control it in Git.
- **Upload UI** is for convenience. When an analyst emails you a spreadsheet and you need to query it quickly, dragging it into Catalog Explorer is faster than writing a CTAS statement. But it's manual and not repeatable.
- In Lesson 02, you used a basic CTAS without any format options: `CREATE TABLE AS SELECT * FROM read_files(...)`. In this lesson, you'll use explicit format options to control how the file is read.
- The format options (`format`, `header`, `inferSchema`) tell `read_files` exactly how to parse the file. This matters when files don't have headers, use a non-standard delimiter, or have ambiguous data types.

</details>

### Explore: Create a table with CTAS and format options

In Lesson 02, you used a basic CTAS: `CREATE TABLE AS SELECT * FROM read_files(...)`. Now let's use explicit format options to control exactly how the file is read.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS current_employees_ctas AS
SELECT ID, FirstName, Country, Role
FROM read_files(
  '/Volumes/' || my_catalog || '/' || my_schema || '/myfiles/employees.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

Notice the differences from Lesson 02:
- **`format => 'csv'`** — explicitly tells Databricks this is a CSV file (instead of auto-detecting)
- **`header => true`** — the first row contains column names, not data
- **`inferSchema => true`** — automatically detect data types for each column
- **`SELECT ID, FirstName, Country, Role`** — we selected only the 4 data columns, excluding the `_rescued_data` column that `read_files` adds automatically

In [0]:
%sql
SELECT * 
FROM current_employees_ctas;

You should see 4 rows with exactly 4 columns. By selecting specific columns in the CTAS, you get a clean table without the extra `_rescued_data` column.

### Explore: Create a table using the Catalog Explorer Upload UI

The second method uses no code at all. You upload a file through the Catalog Explorer interface and Databricks creates the table for you.

**Follow these steps:**

1. In the left sidebar, click **Catalog** to open Catalog Explorer
2. Navigate to your catalog (`labuser`) → your schema (`get_started_de`)
3. Click the **Create** button and select **Table**
4. Drag and drop the `employees.csv` file, or click **browse** to select it
   - If you need to download the file first: go to **Volumes** → **myfiles** → click `employees.csv` → click **Download**
5. Databricks auto-detects the format and infers the schema, review the preview
6. Set the table name to **`current_employees_ui`**
7. Click **Create table**

Once you've created the table, verify it exists by running the cell below.

In [0]:
# Run this after completing the Upload UI steps above.
try:
    display(spark.sql("SELECT * FROM current_employees_ui"))
except Exception as e:
    if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
        print("The table 'current_employees_ui' doesn't exist yet.")
        print("Complete the Upload UI steps above, then re-run this cell.")
    else:
        raise e

### Explore: Compare both tables

Let's verify that both methods produced the same data.

In [0]:
%sql
SHOW TABLES;

You should see both `current_employees_ctas` and `current_employees_ui` listed, along with `employees` from Lesson 02 (and any other tables you created in the practice exercises). Two tables, two methods, same source data.


<!-- Micro-win summary -->

<div style="max-width: 900px; margin: 0 auto; font-family: sans-serif;">
<div style="margin-top: 10px; padding: 18px 24px; background: #FFF6F4; border: 3px solid #FF5F46; border-radius: 10px;">
  <div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">
    <div style="font-weight: 700; margin-bottom: 8px;">What you just did:</div>
    <ul style="padding-left: 20px; margin: 0;">
      <li>Created a table with <code>CTAS</code> using explicit format options (<code>format</code>, <code>header</code>, <code>inferSchema</code>)</li>
      <li>Selected specific columns to produce a clean table without <code>_rescued_data</code></li>
      <li>Created a table using the Catalog Explorer Upload UI with no code</li>
      <li>Verified both methods produced working Delta tables</li>
    </ul>
    <div style="margin-top: 12px;"><strong>CTAS</strong> is code-driven and repeatable. The <strong>Upload UI</strong> is fast but manual.</div>
  </div>
</div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>